# Project 1 — IVI Conceptual Architecture Diagram

**Assignment (Option 1):** Design a conceptual architecture diagram of an IVI system integrating media, navigation, and projection features. Annotate component interfaces and data flow.

This notebook builds the diagram programmatically with Graphviz so it can be regenerated, edited, and re-exported at any resolution — instead of hand-drawing it once.

**Layers modeled:**
1. Application layer — Media Player, Navigation, Projection client apps
2. IVI framework/system services — Media Service, Navigation Service, Projection Service, Telephony Service, Display Compositor
3. Hardware Abstraction Layer (HAL) — Audio HAL, GNSS/Sensor HAL, Display HAL, Vehicle HAL
4. Hardware — speakers, GPS receiver, center display, vehicle CAN bus

Every edge is labeled with the interface/protocol carrying that data (AIDL calls, PCM audio, NMEA fixes, composited surfaces, CAN signals, etc.), satisfying the "annotate interfaces and data flow" requirement.

## 1. Setup (Colab only)
Colab does not ship the Graphviz system binary by default — install it once per runtime.

In [ ]:
!apt-get -qq install -y graphviz > /dev/null
!pip -q install graphviz


## 2. Build the architecture graph

In [ ]:
from graphviz import Digraph


def build_ivi_architecture():
    g = Digraph("IVI_Architecture", format="png")
    g.attr(rankdir="TB", fontsize="11", fontname="Helvetica", splines="polyline")
    g.attr("node", shape="box", style="rounded,filled", fontname="Helvetica", fontsize="10")

    # ---------------- Layer: Applications ----------------
    with g.subgraph(name="cluster_apps") as c:
        c.attr(label="Application Layer", style="dashed", color="gray40")
        c.node("MediaApp", "Media Player App", fillcolor="#AED6F1")
        c.node("NavApp", "Navigation App", fillcolor="#AED6F1")
        c.node("ProjApp", "Projection Client\n(Android Auto / CarPlay)", fillcolor="#AED6F1")

    # ---------------- Layer: Framework / Services ----------------
    with g.subgraph(name="cluster_framework") as c:
        c.attr(label="IVI Framework / System Services", style="dashed", color="gray40")
        c.node("MediaSvc", "Media Service\n(MediaSession, AudioFocus)", fillcolor="#A9DFBF")
        c.node("NavSvc", "Navigation Service\n(Location, Route Engine)", fillcolor="#A9DFBF")
        c.node("ProjSvc", "Projection Service\n(MirrorLink / AAOS Projected)", fillcolor="#A9DFBF")
        c.node("TelephonySvc", "Telephony Service\n(Call State Manager)", fillcolor="#A9DFBF")
        c.node("Compositor", "Display Compositor /\nSurfaceFlinger", fillcolor="#A9DFBF")

    # ---------------- Layer: HAL ----------------
    with g.subgraph(name="cluster_hal") as c:
        c.attr(label="Hardware Abstraction Layer (HAL)", style="dashed", color="gray40")
        c.node("AudioHAL", "Audio HAL", fillcolor="#F9E79F")
        c.node("GNSSHAL", "GNSS / Sensor HAL", fillcolor="#F9E79F")
        c.node("DisplayHAL", "Display HAL", fillcolor="#F9E79F")
        c.node("VehicleHAL", "Vehicle HAL\n(CAN/Bus signals)", fillcolor="#F9E79F")

    # ---------------- Layer: Hardware ----------------
    with g.subgraph(name="cluster_hw") as c:
        c.attr(label="Hardware", style="dashed", color="gray40")
        c.node("Speakers", "Speakers / Amp", fillcolor="#F5B7B1")
        c.node("GPS", "GPS/GNSS Receiver", fillcolor="#F5B7B1")
        c.node("Screen", "Center Display", fillcolor="#F5B7B1")
        c.node("CANBus", "Vehicle CAN Bus", fillcolor="#F5B7B1")

    # ---------------- Edges: data flow with labeled interfaces ----------------
    g.edge("MediaApp", "MediaSvc", label="AIDL: playback control")
    g.edge("NavApp", "NavSvc", label="AIDL: route request")
    g.edge("ProjApp", "ProjSvc", label="USB/Wi-Fi: mirroring stream")

    g.edge("MediaSvc", "AudioHAL", label="audio stream + focus events")
    g.edge("NavSvc", "GNSSHAL", label="location fix request")
    g.edge("ProjSvc", "Compositor", label="rendered frame buffer")
    g.edge("TelephonySvc", "MediaSvc", label="AudioFocus interrupt (call)")
    g.edge("TelephonySvc", "VehicleHAL", label="call state / mic mute")

    g.edge("AudioHAL", "Speakers", label="PCM audio")
    g.edge("GNSSHAL", "GPS", label="NMEA/raw fix")
    g.edge("Compositor", "DisplayHAL", label="composited surface")
    g.edge("DisplayHAL", "Screen", label="pixel data")
    g.edge("VehicleHAL", "CANBus", label="vehicle signals")

    # cross-cutting: navigation also draws turn-by-turn onto compositor
    g.edge("NavSvc", "Compositor", label="map overlay surface", style="dashed", color="gray30")

    return g


## 3. Render and display the diagram

In [ ]:
diagram = build_ivi_architecture()
diagram.render("ivi_architecture", cleanup=True)
from IPython.display import Image, display
display(Image(filename="ivi_architecture.png"))


## 4. Notes on the design

- **Media path:** `MediaApp -> MediaService -> AudioHAL -> Speakers`. The Telephony Service can interrupt this path by asserting an AudioFocus event on Media Service (modeled in Project 2 and Project 4 in more depth).
- **Navigation path:** `NavApp -> NavService -> GNSS HAL -> GPS receiver` for the location fix, and a second, cross-cutting path `NavService -> Compositor` for drawing the turn-by-turn overlay onto the shared display surface.
- **Projection path:** `ProjApp -> ProjectionService -> Compositor -> DisplayHAL -> Screen`, representing an Android Auto/CarPlay style mirrored session sharing the same compositor as native navigation overlays.
- **Interfaces annotated:** AIDL (app-to-service), binder/USB/Wi-Fi transports, HAL-level calls (PCM audio, NMEA, pixel buffers, CAN frames) — showing the full path from a UI tap down to physical hardware.

To extend: add a Bluetooth HAL + hands-free profile box feeding the Telephony Service, or split Projection into separate Android Auto and CarPlay client boxes if the target platform must support both.